# DACON recovery V3 — music specialist comparison

The previous AST candidate failed its promotion gates and underperformed the official baseline. This run preserves PANNs presence and DF-Arena voice detection, and compares five frozen music branches. It performs no training or private-test adaptation.

Development and old challenge results are reused diagnostics. A new source-disjoint challenge accepts or rejects the locked candidate; it does not choose another candidate. Unknown labels remain unknown, and partial metrics are not a DACON leaderboard score. Every input file is processed through its retained duration. A100 allocation is capped at two hours and released on completion or failure.

In [ ]:
import time, json, shutil, subprocess, sys, zipfile, hashlib, threading, traceback, os
from pathlib import Path
from google.colab import drive, runtime
SESSION_START = time.time()
DEADLINE = SESSION_START + 7080
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
WORK = Path('/content/dacon_recovery_v3')
WORK.mkdir(exist_ok=True)
RUN = DRIVE / 'runs_recovery' / time.strftime('%Y%m%d_%H%M%S',time.gmtime(SESSION_START))
RUN.mkdir(parents=True)
watchdog = threading.Timer(max(1,DEADLINE-time.time()),runtime.unassign)
watchdog.daemon = True
watchdog.start()
(RUN/'budget.json').write_text(json.dumps({'start_unix':SESSION_START,'deadline':DEADLINE,'allocation_limit_seconds':7200}))
print('RESULTS_DIRECTORY',RUN,flush=True)

In [ ]:
%%writefile /content/dacon_recovery_v3/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


In [ ]:
%%writefile /content/dacon_recovery_v3/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


In [ ]:
%%writefile /content/dacon_recovery_v3/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


In [ ]:
%%writefile /content/dacon_recovery_v3/v2_scoring.py
"""Exact ROC diagnostics and development-only model selection for V2."""

from __future__ import annotations

import argparse, csv, json, random
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Final

from graph_scoring import official_metrics, validate_predictions

Scalar = str | int | float | None
Row = Mapping[str, Scalar]
PredictionRows = Mapping[str, Sequence[Row]]
WEIGHTS: Final = {"file_eer": .45, "voice_eer": .18, "music_eer": .27, "voice_presence_auc": .05, "music_presence_auc": .05}
EER_SPECS: Final = (("file", "FILE_FAKE", "FILE_FAKE_PROB", None), ("voice", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"), ("music", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"))


def _label(row: Row, name: str) -> int | None:
    value = row.get(name)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise ValueError(f"invalid binary label: {name}")
    return int(value)


def _role(row: Row) -> str:
    role = row.get("role", row.get("split"))
    if not isinstance(role, str) or not role:
        raise ValueError("manifest rows require role or split")
    return role


def _prediction_map(rows: Sequence[Row], manifest: Sequence[Row]) -> dict[str, Row]:
    expected = [str(row["ID"]) for row in manifest]
    parsed = validate_predictions(rows, expected)
    return {row["ID"]: row for row in parsed}


def _eligible(rows: Sequence[Row], predictions: Mapping[str, Row], target: str,
              score: str, mask: str | None) -> tuple[list[int], list[float], int]:
    targets: list[int] = []
    scores: list[float] = []
    unknown = 0
    for row in rows:
        if mask is not None:
            present = _label(row, mask)
            if present is None:
                unknown += 1
                continue
            if present == 0:
                continue
        truth = _label(row, target)
        if truth is None:
            unknown += 1
            continue
        targets.append(truth)
        scores.append(float(predictions[str(row["ID"])][score]))
    return targets, scores, unknown


def _roc(targets: Sequence[int], scores: Sequence[float]) -> dict[str, Scalar] | None:
    positives = sum(targets)
    negatives = len(targets) - positives
    if not positives or not negatives:
        return None
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    points: list[tuple[float | None, int, int]] = [(None, 0, 0)]
    tp = fp = 0
    index = 0
    while index < len(ranked):
        threshold = ranked[index][0]
        while index < len(ranked) and ranked[index][0] == threshold:
            tp += ranked[index][1]
            fp += 1 - ranked[index][1]
            index += 1
        points.append((threshold, tp, fp))
    def distance(point: tuple[float | None, int, int]) -> float:
        return abs(point[2] / negatives - (1 - point[1] / positives))
    threshold, tp, fp = min(points, key=distance)
    fpr, fnr = fp / negatives, 1 - tp / positives
    auc = sum((right[2] / negatives - left[2] / negatives) *
              (right[1] / positives + left[1] / positives) / 2
              for left, right in zip(points, points[1:]))
    return {"eer": (fpr + fnr) / 2, "auc": auc, "threshold": threshold,
            "fpr": fpr, "fnr": fnr, "fp": fp, "fn": positives - tp,
            "tp": tp, "tn": negatives - fp, "n": len(targets)}


def _detail_report(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, dict[str, Scalar]]:
    details: dict[str, dict[str, Scalar]] = {}
    for name, target, score, mask in EER_SPECS:
        targets, scores, unknown = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        details[name] = {"unknown_relevant": unknown, "n": len(targets),
                         "point": point}
    return details


def _pair_changes(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row],
                  target: str, score: str, mask: str | None) -> dict[str, int]:
    labels, scores_a, _ = _eligible(rows, baseline, target, score, mask)
    _, scores_b, _ = _eligible(rows, candidate, target, score, mask)
    positives = [index for index, value in enumerate(labels) if value == 1]
    negatives = [index for index, value in enumerate(labels) if value == 0]
    flips = worsened = 0
    for positive in positives:
        for negative in negatives:
            before = scores_a[positive] - scores_a[negative]
            after = scores_b[positive] - scores_b[negative]
            if (before > 0 > after) or (before < 0 < after):
                flips += 1
                worsened += int(before > 0 > after)
    return {"pairs": len(positives) * len(negatives), "flips": flips,
            "positive_to_negative_worsened": worsened}


def audit_forensics(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Report all available roles without converting a partial proxy into a score."""
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for forensics")
    roles: dict[str, list[Row]] = defaultdict(list)
    for row in manifest_rows:
        roles[_role(row)].append(row)
    report: dict[str, object] = {"metric": "nearest_abs_fpr_minus_fnr", "roles": {}}
    for role, rows in sorted(roles.items()):
        models: dict[str, object] = {}
        for name, prediction in maps.items():
            models[name] = {"official_proxy_only": official_metrics(rows, [prediction[str(row["ID"])] for row in rows]),
                            "roc": _detail_report(rows, prediction)}
        comparisons: dict[str, object] = {}
        for name, prediction in maps.items():
            if name == "G0":
                continue
            per_head: dict[str, object] = {}
            for head, target, score, mask in EER_SPECS:
                base = models["G0"]["roc"][head]["point"]
                changed = models[name]["roc"][head]["point"]
                per_head[head] = {"baseline": base, "candidate": changed,
                                  "auc_delta": None if base is None or changed is None else changed["auc"] - base["auc"],
                                  "pair_changes": _pair_changes(rows, maps["G0"], prediction, target, score, mask)}
            comparisons[name] = per_head
        report["roles"][role] = {"models": models, "comparisons": comparisons}
    return report


def _proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, object]:
    metrics = official_metrics(rows, [predictions[str(row["ID"])] for row in rows])
    usable = {name: weight for name, weight in WEIGHTS.items() if metrics["metrics"][name] is not None}
    total = sum(usable.values())
    values = metrics["metrics"]
    proxy = None if not total else sum(weight / total * ((1 - float(values[name])) if name.endswith("eer") else float(values[name]))
                                      for name, weight in usable.items())
    return {"value": proxy, "proxy_weights": usable, "metric_report": metrics,
            "is_partial": bool(metrics["partial"])}


def _resampled_proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> float | None:
    """Score a bootstrap sample whose repeated IDs cannot enter CSV validation."""
    values: dict[str, float] = {}
    for metric, target, score, mask in EER_SPECS:
        targets, scores, _ = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_eer"] = point["eer"]
    for metric, target, score in (("voice_presence", "VOICE_PRESENT", "VOICE_PRESENT_PROB"),
                                  ("music_presence", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB")):
        targets, scores, _ = _eligible(rows, predictions, target, score, None)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_auc"] = point["auc"]
    usable = {name: weight for name, weight in WEIGHTS.items() if name in values}
    total = sum(usable.values())
    return None if not total else sum(weight / total * ((1 - values[name]) if name.endswith("eer") else values[name])
                                     for name, weight in usable.items())


def _components(rows: Sequence[Row]) -> list[list[int]]:
    parent: dict[str, str] = {}
    def find(value: str) -> str:
        parent.setdefault(value, value)
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value
    def join(left: str, right: str) -> None:
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left
    for row in rows:
        raw_group = row.get("group_id")
        if not isinstance(raw_group, str) or not raw_group:
            raise ValueError("manifest rows require nonempty group_id constituents")
        groups = raw_group.split("|")
        if any(not group for group in groups):
            raise ValueError("manifest rows require nonempty group_id constituents")
        for group in groups[1:]:
            join(groups[0], group)
    units: dict[str, list[int]] = defaultdict(list)
    for index, row in enumerate(rows):
        units[find(str(row["group_id"]).split("|")[0])].append(index)
    return list(units.values())


def _bootstrap(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> dict[str, object]:
    units = _components(rows)
    rng = random.Random(20260913)
    deltas: list[float] = []
    for _ in range(2000):
        sampled = [row for unit in (rng.choice(units) for _ in units) for index in unit for row in [rows[index]]]
        base, changed = _resampled_proxy(sampled, baseline), _resampled_proxy(sampled, candidate)
        if base is not None and changed is not None:
            deltas.append(changed - base)
    deltas.sort()
    return {"seed": 20260913, "resamples": 2000, "lineage_components": len(units),
            "valid_resamples": len(deltas), "uncomputable_resamples": 2000 - len(deltas),
            "win_rate": None if not deltas else sum(delta > 0 for delta in deltas) / len(deltas),
            "p10_delta": None if not deltas else deltas[max(0, (len(deltas) - 1) // 10)]}


def select_candidate(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Select a V2 candidate only from the dedicated development selector role."""
    if any(_role(row) != "dev_select" for row in manifest_rows):
        raise ValueError("selection accepts development roles only")
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for selection")
    models: dict[str, dict[str, object]] = {"G0": _proxy(manifest_rows, maps["G0"])}
    candidates = [name for name in ("AST_FROZEN", "AST_LAST_BLOCK", "G0_AST_MUSIC") if name in maps]
    for name in candidates:
        current = _proxy(manifest_rows, maps[name])
        bootstrap = _bootstrap(manifest_rows, maps["G0"], maps[name])
        current["bootstrap"] = bootstrap
        base_metrics = models["G0"]["metric_report"]["metrics"]
        new_metrics = current["metric_report"]["metrics"]
        regression = any(float(new_metrics[key]) > float(base_metrics[key]) + .02 for key in WEIGHTS if key.endswith("eer") and new_metrics[key] is not None and base_metrics[key] is not None)
        companion = any(_roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2])["auc"] < _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2])["auc"] - .01 for _, target, score, mask in EER_SPECS if _roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2]) is not None and _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2]) is not None)
        gain = None if current["value"] is None or models["G0"]["value"] is None else float(current["value"]) - float(models["G0"]["value"])
        gates = {"proxy_gain": gain is not None and gain >= .005,
                 "bootstrap_complete": bootstrap["valid_resamples"] == 2000,
                 "bootstrap_win_rate": bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
                 "bootstrap_p10": bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005,
                 "eer_regression": not regression, "companion_auc_regression": not companion}
        current["gates"] = {**gates, "proxy_gain_delta": gain, "passed": all(gates.values())}
        models[name] = current
    passed = [name for name in candidates if models[name]["gates"]["passed"]]
    selected = max(passed, key=lambda name: (models[name]["value"], -candidates.index(name)), default="G0")
    failed = {name: [gate for gate, value in models[name]["gates"].items() if value is False]
              for name in candidates if name not in passed}
    return {"selected": selected, "reason": "all_gates_passed" if selected != "G0" else "no_candidate_passed_all_gates",
            "failed_gates": failed, "models": models}


def _read_csv(path: Path) -> list[dict[str, Scalar]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return [dict(row) for row in csv.DictReader(stream)]


def _load_predictions(directory: Path, manifest: Sequence[Row]) -> dict[str, list[Row]]:
    files = {name: directory / f"{name}_predictions.csv" for name in ("G0", "G1", "G2", "G3", "G4")}
    if all(path.is_file() for path in files.values()):
        return {name: _read_csv(path) for name, path in files.items()}
    from graph_fusion import graph_predictions
    features = json.loads((directory / "features.json").read_text())
    fusion = json.loads((directory / "fusion.json").read_text())
    by_id = {feature["ID"]: graph_predictions(feature, fusion) for feature in features}
    return {name: [by_id[str(row["ID"])][name] for row in manifest] for name in files}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("diagnose")
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--predictions", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    manifest = _read_csv(args.manifest)
    report = audit_forensics(manifest, _load_predictions(args.predictions, manifest))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_recovery_v3/run_experiment.py
"""A bounded pilot; every GPU result is checkpointed to the mounted Drive."""

import argparse
import csv
import importlib.metadata
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from graph_features import Extractor, atomic_json, extract_cached
from graph_fusion import fit_fusion, graph_predictions
from graph_scoring import official_metrics, validate_predictions


def read_csv(path: Path) -> list:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list) -> None:
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def reference_parity(extractor: Extractor, path: Path, feature) -> dict:
    """Compare the shared graph with untouched official functions on real audio."""
    base = extractor.base
    audio = base.load_audio(path)
    vp, mp = base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
    extractor.demucs.cpu()
    voice, music = base.separate_voice_and_music(path, extractor.demucs, extractor.device)
    extractor.demucs.to(extractor.device)
    vf = base.predict_fake(extractor.df, extractor.fake_index, voice, extractor.device)
    mf = base.predict_fake(extractor.df, extractor.fake_index, music, extractor.device)
    expected = np.array([base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp])
    actual = np.array(list(graph_predictions(feature)['G0'].values())[1:])
    delta = float(np.max(np.abs(expected - actual)))
    if not np.allclose(expected, actual, rtol=1e-5, atol=1e-6):
        raise AssertionError(f'G0 differs from official inference by {delta}: {path}')
    return {'ID': path.stem, 'max_probability_delta': delta, 'reference': expected.tolist()}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--baseline', type=Path, required=True)
    parser.add_argument('--validation', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--deadline', type=float, required=True)
    args = parser.parse_args()
    out = args.output
    out.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if start >= args.deadline:
        raise TimeoutError('The existing GPU budget has expired; no new run was started')
    labels = read_csv(args.validation / 'manifest.csv')
    if len({r['ID'] for r in labels}) != len(labels):
        raise ValueError('Duplicate validation IDs')
    paths = {r['ID']: args.validation / r['path'] for r in labels}
    if any(p.stem != key or not p.is_file() for key, p in paths.items()):
        raise ValueError('Manifest ID/path mismatch or missing audio')
    atomic_json(out / 'environment.json', {
        'python': sys.version, 'gpu': torch.cuda.get_device_name(),
        'gpu_total_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'started_unix': start, 'deadline_unix': args.deadline,
        'packages': {k: importlib.metadata.version(k) for k in
            ('torch', 'torchaudio', 'transformers', 'numpy', 'scipy', 'scikit-learn', 'demucs', 'librosa')},
    })
    extractor = Extractor(args.baseline / 'script.py')
    try:
        dummy = sorted((args.baseline / 'data' / 'test').glob('*.wav'))
        dummy_features = extract_cached(extractor, dummy, out / 'cache', args.deadline)
        atomic_json(out / 'dummy_features.json', dummy_features)
        ordered = list(paths.values())
        random.Random(20260913).shuffle(ordered)
        first = extract_cached(extractor, ordered[:1], out / 'cache', args.deadline)
        if not first:
            raise TimeoutError('No validation feature completed within the budget')
        atomic_json(out / 'reference_parity.json', reference_parity(extractor, ordered[0], first[0]))
        stress_path = args.baseline / 'stress_60s.wav'
        signal = extractor.base.load_audio(ordered[0])
        sf.write(stress_path, np.tile(signal, 960000 // len(signal) + 1)[:960000], 16000)
        try:
            stress = extract_cached(extractor, [stress_path], out / 'stress_cache', args.deadline)
            atomic_json(out / 'stress60.json', {'completed': bool(stress),
                'purpose': '60-second runtime and memory only; repeated source is not validation evidence'})
        except torch.cuda.OutOfMemoryError as error:
            atomic_json(out / 'stress60.json', {'completed': False, 'error': str(error),
                'submission_supported': False})
            torch.cuda.empty_cache()
        features = extract_cached(extractor, ordered, out / 'cache', args.deadline)
    finally:
        extractor.close()
    atomic_json(out / 'features.json', features)
    available = {f['ID'] for f in features}
    completed_labels = [r for r in labels if r['ID'] in available]
    fusion = fit_fusion(features, labels)
    atomic_json(out / 'fusion.json', fusion)
    predictions = {name: [] for name in ('G0', 'G1', 'G2', 'G3', 'G4')}
    for feature in features:
        for name, row in graph_predictions(feature, fusion).items():
            predictions[name].append(row)
    report = {'complete': len(features) == len(labels), 'completed': len(features),
        'planned': len(labels), 'elapsed_seconds': time.time() - start,
        'scope': 'external_source_group_holdout_not_DACON_leaderboard', 'graphs': {}}
    heldout = [r for r in completed_labels if r['split'] == 'holdout']
    heldout_ids = {r['ID'] for r in heldout}
    heldout_groups = {g for r in heldout for g in r['group_id'].split('|')}
    report['holdout'] = {'clips': len(heldout), 'atomic_source_groups': len(heldout_groups),
        'limitation': 'Transforms and mixtures are correlated; source groups are the independent units'}
    weights = {'file_eer': .45, 'voice_eer': .18, 'music_eer': .27,
        'voice_presence_auc': .05, 'music_presence_auc': .05}
    for name, rows in predictions.items():
        validate_predictions(rows, [f['ID'] for f in features])
        write_csv(out / f'{name}_predictions.csv', rows)
        by_id = {r['ID']: r for r in rows if r['ID'] in heldout_ids}
        selected = [by_id[r['ID']] for r in heldout]
        result = official_metrics(heldout, selected) if heldout else {
            'score': None, 'metrics': {k: None for k in weights},
            'invalid': {k: 'no completed holdout examples' for k in weights},
            'partial': {}, 'counts': {k: 0 for k in weights},
            'unknown_counts': {k: 0 for k in weights}, 'coverage': {k: 0.0 for k in weights}}
        # ponytail: partial-label proxy only selects this pilot; it is never an official score.
        result['selection_proxy'] = sum(weight * ((1 - result['metrics'][metric]) if metric.endswith('eer')
            else result['metrics'][metric]) for metric, weight in weights.items()
            if result['metrics'][metric] is not None)
        report['graphs'][name] = result
        print(name, json.dumps(result, ensure_ascii=False), flush=True)
    ranked = sorted(report['graphs'], key=lambda g: report['graphs'][g]['selection_proxy'], reverse=True)
    best = ranked[0]
    improvement = report['graphs'][best]['selection_proxy'] - report['graphs']['G0']['selection_proxy']
    stress_ok = json.loads((out / 'stress60.json').read_text())['completed']
    eligible = report['complete'] and stress_ok and len(heldout_groups) >= 20 and improvement >= .01
    report['selection'] = {'graph': best if eligible else 'G0', 'best_observed_graph': best,
        'proxy_improvement_over_G0': improvement, 'new_submission_supported': eligible,
        'reason': 'complete_pilot_and_proxy_gain_at_least_0.01' if eligible else 'retain_reference_without_sufficient_evidence'}
    records = [json.loads(p.read_text()) for p in (out / 'cache').glob('*.json')]
    timings = [r['seconds'] for r in records]
    report['runtime'] = {'unique_cached_files': len(records), 'total_extraction_seconds': sum(timings),
        'mean_seconds_per_file': float(np.mean(timings)),
        'max_allocated_gib': max(r['max_allocated_gib'] for r in records),
        'max_reserved_gib': max(r['max_reserved_gib'] for r in records),
        'L4_60_minute_compliance': 'unverified; A100 measurements are not L4 evidence'}
    report['runtime']['stress60'] = [
        {k: v for k, v in json.loads(p.read_text()).items() if k != 'feature'}
        for p in (out / 'stress_cache').glob('*.json')]
    atomic_json(out / 'comparison.json', report)
    for name in predictions:
        dummy_rows = [graph_predictions(f, fusion)[name] for f in dummy_features]
        write_csv(out / f'{name}_dummy_submission.csv', dummy_rows)
    atomic_json(out / 'status.json', {'status': 'comparison_finished', 'unix': time.time(),
        'selected_graph': report['selection']['graph'], 'completed': len(features)})
    print('COMPARISON_FINISHED', json.dumps(report['selection']), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/dacon_recovery_v3/recovery_sonics.py
"""Pinned, file-local SONICS music-authenticity inference."""

from __future__ import annotations

import hashlib
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Final, TypedDict

import numpy as np
import torch

SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = SAMPLE_RATE * 5
CONFIG_SHA256: Final = "754bc15b3f914afaf4e002e50bfaa5e81055c9fb0e2b76408e52fd88f09f7def"
WEIGHTS_SHA256: Final = "11c384cea651c56028d6f02e04aa467c2099cf1102687353fe4dd9504c4db868"


class SonicsPrediction(TypedDict):
    mean_probability: float
    window_probabilities: list[float]
    starts: list[int]
    covered_samples: int
    seconds: float
    audio_seconds: float
    elapsed_seconds: float


@dataclass(frozen=True, slots=True)
class SonicsContractError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _sha256(path: Path) -> str:
    """Return one asset's full SHA-256 digest."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _verified_assets(assets: Path) -> None:
    """Reject a missing or altered local asset before checkpoint deserialization."""
    for name, expected in (("config.json", CONFIG_SHA256), ("pytorch_model.bin", WEIGHTS_SHA256)):
        path = assets / name
        if not path.is_file():
            raise SonicsContractError(f"SONICS local asset is missing: {path}")
        if _sha256(path) != expected:
            raise SonicsContractError(f"SONICS {name} SHA-256 mismatch")


class SonicsPredictor:
    """Run the pinned classifier serially over one 16 kHz mono waveform."""

    model: torch.nn.Module
    device: torch.device

    def __init__(self, assets: Path, device: str) -> None:
        _verified_assets(assets)
        vendor_root = assets.parent / "sonics_vendor"
        if not (vendor_root / "sonics").is_dir():
            raise SonicsContractError(f"Pinned SONICS source is missing beside assets: {vendor_root}")
        sys.path.insert(0, str(vendor_root))
        from sonics.models.hf_model import HFAudioClassifier

        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.device = torch.device(device)
        self.model = HFAudioClassifier.from_pretrained(str(assets), map_location="cpu", strict=True).to(self.device).eval()

    def predict(self, audio: np.ndarray) -> SonicsPrediction:
        """Return a mean of serial 5-second window probabilities for one file."""
        waveform = np.asarray(audio)
        if waveform.ndim != 1 or waveform.size == 0:
            raise SonicsContractError("SONICS audio must be nonempty and one-dimensional")
        try:
            waveform = np.ascontiguousarray(waveform, dtype=np.float32)
        except (TypeError, ValueError) as error:
            raise SonicsContractError("SONICS audio must be convertible to float32") from error
        if not np.isfinite(waveform).all():
            raise SonicsContractError("SONICS audio must contain only finite values")
        starts = self._window_starts(waveform.size)
        started = time.perf_counter()
        probabilities = [self._window_probability(waveform[start : start + WINDOW_SAMPLES]) for start in starts]
        elapsed = time.perf_counter() - started
        if not all(math.isfinite(value) and 0.0 <= value <= 1.0 for value in probabilities):
            raise SonicsContractError("SONICS model returned a non-finite probability")
        covered_samples = self._covered_samples(starts, waveform.size)
        if covered_samples != waveform.size:
            raise SonicsContractError("SONICS windows must cover every original sample")
        audio_seconds = waveform.size / SAMPLE_RATE
        return {
            "mean_probability": float(np.mean(probabilities)),
            "window_probabilities": probabilities,
            "starts": starts,
            "covered_samples": covered_samples,
            "seconds": audio_seconds,
            "audio_seconds": audio_seconds,
            "elapsed_seconds": elapsed,
        }

    @staticmethod
    def _window_starts(samples: int) -> list[int]:
        """Cover the tail with a final full window; only short files are padded."""
        if samples <= WINDOW_SAMPLES:
            return [0]
        final_start = samples - WINDOW_SAMPLES
        starts = list(range(0, final_start + 1, WINDOW_SAMPLES))
        if starts[-1] != final_start:
            starts.append(final_start)
        return starts

    @staticmethod
    def _covered_samples(starts: list[int], samples: int) -> int:
        """Measure the covered union, rejecting a gap rather than reporting input length."""
        covered_end = 0
        for start in starts:
            if start > covered_end:
                return covered_end
            covered_end = max(covered_end, min(samples, start + WINDOW_SAMPLES))
        return covered_end

    def _window_probability(self, samples: np.ndarray) -> float:
        """Infer exactly one window, preserving per-example feature normalization."""
        padded = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        padded[: samples.size] = samples
        with torch.no_grad():
            logits = self.model(torch.from_numpy(padded).unsqueeze(0).to(self.device))
        if logits.numel() != 1:
            raise SonicsContractError("SONICS model must return exactly one logit per window")
        logit = float(logits.item())
        if not math.isfinite(logit):
            raise SonicsContractError("SONICS model returned a non-finite logit")
        return float(torch.sigmoid(torch.tensor(logit)).item())


In [ ]:
%%writefile /content/dacon_recovery_v3/recovery_fusion.py
"""The five fixed, file-local Recovery V3 candidate formulas."""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Final

from graph_fusion import Prediction

CANDIDATES: Final = ("G0", "SONICS_RAW", "SONICS_STEM", "BLEND_RAW", "BLEND_STEM")


@dataclass(frozen=True, slots=True)
class FusionContractError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _probability(value: float, name: str) -> float:
    """Parse one probability at the file-local fusion boundary."""
    try:
        parsed = float(value)
    except (TypeError, ValueError) as error:
        raise FusionContractError(f"{name} must be a finite probability") from error
    if not math.isfinite(parsed) or not 0.0 <= parsed <= 1.0:
        raise FusionContractError(f"{name} must be a finite probability in [0, 1]")
    return parsed


def _row(g0: Prediction, music: float) -> Prediction:
    """Keep G0's voice and presence heads exact while replacing music authenticity."""
    voice_fake = _probability(g0["VOICE_FAKE_PROB"], "G0 voice fake probability")
    voice_present = _probability(g0["VOICE_PRESENT_PROB"], "G0 voice presence probability")
    music_present = _probability(g0["MUSIC_PRESENT_PROB"], "G0 music presence probability")
    return {
        "ID": g0["ID"],
        "FILE_FAKE_PROB": max(voice_fake * voice_present, music * music_present),
        "VOICE_FAKE_PROB": g0["VOICE_FAKE_PROB"],
        "MUSIC_FAKE_PROB": music,
        "VOICE_PRESENT_PROB": g0["VOICE_PRESENT_PROB"],
        "MUSIC_PRESENT_PROB": g0["MUSIC_PRESENT_PROB"],
    }


def candidate_predictions(g0: Prediction, raw_prob: float, stem_prob: float) -> dict[str, Prediction]:  # noqa: DICT_OK
    """Build exactly the declared candidates for one file without cross-file state."""
    if not isinstance(g0["ID"], str) or not g0["ID"]:
        raise FusionContractError("G0 ID must be a nonempty string")
    baseline = {name: _probability(g0[name], f"G0 {name}") for name in (
        "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB"
    )}
    raw = _probability(raw_prob, "SONICS raw probability")
    stem = _probability(stem_prob, "SONICS stem probability")
    g0_row: Prediction = {"ID": g0["ID"], **baseline}
    music = baseline["MUSIC_FAKE_PROB"]
    return {
        "G0": g0_row,
        "SONICS_RAW": _row(g0_row, raw),
        "SONICS_STEM": _row(g0_row, stem),
        "BLEND_RAW": _row(g0_row, 0.5 * music + 0.5 * raw),
        "BLEND_STEM": _row(g0_row, 0.5 * music + 0.5 * stem),
    }


In [ ]:
%%writefile /content/dacon_recovery_v3/recovery_selection.py
"""Frozen candidate selection and one-way fresh acceptance for Recovery V3."""

from __future__ import annotations

import hashlib
import json
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Final

from graph_scoring import validate_predictions
from v2_scoring import _bootstrap, _detail_report, _proxy

Scalar = str | int | float | None
Row = Mapping[str, Scalar]
PredictionSets = Mapping[str, Sequence[Row]]
CANDIDATES: Final = ("G0", "SONICS_RAW", "SONICS_STEM", "BLEND_RAW", "BLEND_STEM")
PROTECTED: Final = ("VOICE_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB")
EVALUATED_HEADS: Final = ("file", "music")


@dataclass(frozen=True, slots=True)
class SelectionError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _maps(rows: Sequence[Row], prediction_sets: PredictionSets) -> dict[str, dict[str, Row]]:
    if tuple(prediction_sets) != CANDIDATES:
        raise SelectionError(f"candidate registry must be exactly {CANDIDATES}")
    expected = [str(row["ID"]) for row in rows]
    maps = {name: {row["ID"]: row for row in validate_predictions(values, expected)}
            for name, values in prediction_sets.items()}
    baseline = maps["G0"]
    for name in CANDIDATES[1:]:
        for identifier in expected:
            g0, candidate = baseline[identifier], maps[name][identifier]
            if any(candidate[field] != g0[field] for field in PROTECTED):
                raise SelectionError(f"protected G0 field changed: {name}/{identifier}")
            file_fake = max(float(g0["VOICE_PRESENT_PROB"]) * float(g0["VOICE_FAKE_PROB"]),
                            float(g0["MUSIC_PRESENT_PROB"]) * float(candidate["MUSIC_FAKE_PROB"]))
            if float(candidate["FILE_FAKE_PROB"]) != file_fake:
                raise SelectionError(f"FILE formula mismatch: {name}/{identifier}")
    return maps


def _report(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, Scalar | dict[str, Scalar]]:
    proxy = _proxy(rows, predictions)
    metrics = proxy["metric_report"]
    details = _detail_report(rows, predictions)
    return {"known_subset_weighted_diagnostic": proxy["value"],
            "weights_used": proxy["proxy_weights"], "metrics": metrics["metrics"],
            "counts": metrics["counts"], "unknown_counts": metrics["unknown_counts"],
            "coverage": metrics["coverage"], "partial": metrics["partial"],
            "invalid": metrics["invalid"], "companion_auc": {
                head: details[head]["point"]["auc"] if details[head]["point"] is not None else None
                for head in EVALUATED_HEADS}}


def _relative_gates(base: Mapping[str, Scalar | dict[str, Scalar]],
                    current: Mapping[str, Scalar | dict[str, Scalar]], gain_required: bool) -> dict[str, Scalar]:
    base_metrics, metrics = base["metrics"], current["metrics"]
    base_auc, auc = base["companion_auc"], current["companion_auc"]
    gain = None
    if base["known_subset_weighted_diagnostic"] is not None and current["known_subset_weighted_diagnostic"] is not None:
        gain = float(current["known_subset_weighted_diagnostic"]) - float(base["known_subset_weighted_diagnostic"])
    gates: dict[str, Scalar] = {"diagnostic_gain": not gain_required or (gain is not None and gain >= .005),
        "diagnostic_gain_delta": gain}
    for head in EVALUATED_HEADS:
        eer_name = f"{head}_eer"
        gates[f"{head}_eer"] = (metrics[eer_name] is not None and base_metrics[eer_name] is not None
            and float(metrics[eer_name]) <= float(base_metrics[eer_name]) + .02)
        gates[f"{head}_companion_auc"] = (auc[head] is not None and base_auc[head] is not None
            and float(auc[head]) >= float(base_auc[head]) - .01)
    gates["passed"] = all(value is True for key, value in gates.items() if key != "diagnostic_gain_delta")
    return gates


def evaluate_old(dev_rows: Sequence[Row], old_rows: Sequence[Row], prediction_sets: PredictionSets) -> dict[str, Scalar | dict]:
    """Rank on dev only; reused old-domain roles can only veto candidates."""
    if any(row.get("split") != "dev_select" for row in dev_rows):
        raise SelectionError("development selection requires dev_select rows only")
    if any(row.get("split") not in ("test_generator", "test_source") for row in old_rows):
        raise SelectionError("old safety rows require test_generator/test_source")
    rows = [*dev_rows, *old_rows]
    maps = _maps(rows, prediction_sets)
    dev_ids = [str(row["ID"]) for row in dev_rows]
    dev_maps = {name: {identifier: values[identifier] for identifier in dev_ids} for name, values in maps.items()}
    base = _report(dev_rows, dev_maps["G0"])
    candidates: dict[str, dict] = {"G0": {"development": base}}
    passed: list[str] = []
    for name in CANDIDATES[1:]:
        development = _report(dev_rows, dev_maps[name])
        bootstrap = _bootstrap(dev_rows, dev_maps["G0"], dev_maps[name])
        dev_gates = _relative_gates(base, development, True)
        dev_gates.update(bootstrap_complete=bootstrap["valid_resamples"] == 2000,
            bootstrap_win_rate=bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
            bootstrap_p10=bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005)
        dev_gates["passed"] = all(value is True for key, value in dev_gates.items()
                                      if key not in ("diagnostic_gain_delta", "passed"))
        slices: dict[str, dict] = {}
        for role in ("test_generator", "test_source"):
            selected = [row for row in old_rows if row["split"] == role]
            if selected:
                slices[role] = _relative_gates(_report(selected, maps["G0"]), _report(selected, maps[name]), False)
        failed_slices = [role for role, value in slices.items() if not value["passed"]]
        old_safety = {"usage": "REUSED diagnostic safety veto; never ranking", "slices": slices,
                      "failed_slices": failed_slices, "passed": bool(slices) and not failed_slices}
        candidates[name] = {"development": development, "development_gates": dev_gates,
                            "bootstrap": bootstrap, "old_safety": old_safety}
        if dev_gates["passed"]:
            passed.append(name)
    development_selected = max(passed, key=lambda name: (float(candidates[name]["development"]["known_subset_weighted_diagnostic"]),
                                                          -CANDIDATES.index(name)), default="G0")
    selected = development_selected
    reason = "all_gates_passed"
    if development_selected == "G0":
        reason = "no_candidate_passed_development_gates"
    elif not candidates[development_selected]["old_safety"]["passed"]:
        selected, reason = "G0", "development_winner_failed_old_safety_veto"
    return {"development_selected": development_selected, "selected": selected, "reason": reason,
            "old_usage": "REUSED diagnostic safety veto; never ranking", "candidates": candidates}


def lock_candidate(path: Path, payload: Mapping[str, Scalar | dict]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    serialized = json.dumps(payload, sort_keys=True, indent=2, allow_nan=False) + "\n"
    if path.is_file():
        if path.read_text(encoding="utf-8") != serialized:
            raise SelectionError("candidate lock is immutable")
        return _digest(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(serialized, encoding="utf-8")
    temporary.replace(path)
    return _digest(path)


def _fresh_maps(rows: Sequence[Row], prediction_sets: PredictionSets, selected: str) -> dict[str, dict[str, Row]]:
    names = tuple(prediction_sets)
    if names == CANDIDATES:
        return _maps(rows, prediction_sets)
    expected_names = ("G0",) if selected == "G0" else ("G0", selected)
    if names != expected_names:
        raise SelectionError("fresh predictions must contain G0 and the frozen candidate only")
    expected = [str(row["ID"]) for row in rows]
    maps = {name: {row["ID"]: row for row in validate_predictions(values, expected)}
            for name, values in prediction_sets.items()}
    if selected == "G0":
        return maps
    for identifier in expected:
        g0, candidate = maps["G0"][identifier], maps[selected][identifier]
        if any(candidate[field] != g0[field] for field in PROTECTED):
            raise SelectionError(f"protected G0 field changed: {selected}/{identifier}")
        expected_file = max(float(g0["VOICE_PRESENT_PROB"]) * float(g0["VOICE_FAKE_PROB"]),
                            float(g0["MUSIC_PRESENT_PROB"]) * float(candidate["MUSIC_FAKE_PROB"]))
        if float(candidate["FILE_FAKE_PROB"]) != expected_file:
            raise SelectionError(f"FILE formula mismatch: {selected}/{identifier}")
    return maps


def decide_fresh(lock_path: Path, manifest_path: Path, rows: Sequence[Row],
                 prediction_sets: PredictionSets) -> dict[str, Scalar | list[str] | dict]:
    """Accept or reject the frozen candidate without selecting on fresh labels."""
    lock = json.loads(lock_path.read_text(encoding="utf-8"))
    if _digest(manifest_path) != lock["fresh_manifest_sha256"]:
        raise SelectionError("fresh manifest changed after candidate lock")
    selected = lock["selected"]
    if selected not in CANDIDATES:
        raise SelectionError("locked candidate is outside the registry")
    maps = _fresh_maps(rows, prediction_sets, selected)
    ignored = [name for name in CANDIDATES if name not in ("G0", selected)]
    if selected == "G0":
        baseline = _report(rows, maps["G0"])
        return {"final": "G0", "accepted": False, "reason": "development_selected_G0",
                "evaluated": ["G0"], "ignored_candidates": ignored,
                "fresh": {"G0": baseline, "scope": "diagnostic_only_no_promotion"}}
    base, current = _report(rows, maps["G0"]), _report(rows, maps[selected])
    gates = _relative_gates(base, current, True)
    bootstrap = _bootstrap(rows, maps["G0"], maps[selected])
    gates.update(bootstrap_complete=bootstrap["valid_resamples"] == 2000,
        bootstrap_win_rate=bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
        bootstrap_p10=bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005)
    provider_slices: dict[str, dict[str, Scalar]] = {}
    generated_sources = sorted({str(row["source"]) for row in rows
        if row.get("MUSIC_FAKE") in (1, "1") and row.get("MUSIC_PRESENT") in (1, "1")})
    for source in generated_sources:
        generated = [row for row in rows if row.get("source") == source
                     and row.get("MUSIC_FAKE") in (1, "1")]
        groups = {part for row in generated for part in str(row["group_id"]).split("|")}
        matching_real = [row for row in rows if row.get("MUSIC_FAKE") in (0, "0")
                         and groups.intersection(str(row["group_id"]).split("|"))]
        family = [*matching_real, *generated]
        known = {int(row["MUSIC_FAKE"]) for row in family if row.get("MUSIC_FAKE") in (0, 1, "0", "1")}
        if known == {0, 1}:
            provider_slices[source] = _relative_gates(_report(family, maps["G0"]), _report(family, maps[selected]), False)
    provider_passed = all(value["passed"] for value in provider_slices.values())
    gates["provider_families"] = provider_passed
    gates["passed"] = all(value is True for key, value in gates.items()
                               if key not in ("diagnostic_gain_delta", "passed"))
    accepted = bool(gates["passed"])
    return {"final": selected if accepted else "G0", "accepted": accepted,
            "reason": "frozen_candidate_accepted" if accepted else "frozen_candidate_failed_fresh_gate",
            "evaluated": ["G0", selected], "ignored_candidates": ignored,
            "fresh": {"G0": base, selected: current, "gates": gates, "bootstrap": bootstrap,
                      "provider_slices": provider_slices}}


In [ ]:
%%writefile /content/dacon_recovery_v3/run_recovery.py
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.11"
# dependencies = ["numpy", "soundfile", "torch"]
# ///
# How to run: uv run run_recovery.py --help
"""Run the bounded, inference-only Recovery V3 comparison."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from itertools import pairwise
from pathlib import Path
from typing import TypeVar

import numpy as np
import soundfile as sf
import torch

from graph_features import Extractor, atomic_json, digest
from recovery_fusion import CANDIDATES, candidate_predictions
from recovery_selection import decide_fresh, evaluate_old, lock_candidate
from recovery_sonics import SonicsPredictor

T = TypeVar("T")
RecoveryRecord = dict[str, str | dict]


@dataclass(frozen=True, slots=True)
class RecoveryError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: Sequence[Mapping[str, str | float]]) -> None:
    if not rows:
        raise RecoveryError(f"refusing empty CSV: {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def synchronized(device: torch.device, action: Callable[[], T]) -> tuple[T, float]:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    value = action()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return value, time.perf_counter() - started


def runtime_signature(extractor: Extractor, baseline: Path, assets: Path) -> tuple[str, dict[str, str]]:
    files = [Path(__file__), Path("recovery_fusion.py"), Path("recovery_selection.py"),
             Path("recovery_sonics.py"), baseline / "script.py", assets / "config.json",
             assets / "pytorch_model.bin"]
    hashes = {str(path): digest(path) for path in files}
    vendor_candidates = (assets.parent / "sonics_vendor",
        Path("_workspace/recovery_v3/vendor/sonics-9156ffad151f797c71556923c4a02fa01fa8fc91"))
    vendor = next((path for path in vendor_candidates if path.is_dir()), None)
    if vendor is None:
        raise RecoveryError("pinned SONICS vendor source is missing")
    hashes.update({str(path): digest(path) for path in sorted(vendor.rglob("*.py"))})
    return hashlib.sha256((extractor.signature + json.dumps(hashes, sort_keys=True)).encode()).hexdigest(), hashes


def _g0(identifier: str, presence: np.ndarray, voice: Sequence[float], music: Sequence[float]) -> dict[str, str | float]:
    vp, mp = presence.max(axis=0)
    vf, mf = max(voice), max(music)
    return {"ID": identifier, "FILE_FAKE_PROB": round(max(vf * vp, mf * mp), 10),
            "VOICE_FAKE_PROB": round(vf, 10), "MUSIC_FAKE_PROB": round(mf, 10),
            "VOICE_PRESENT_PROB": round(float(vp), 10), "MUSIC_PRESENT_PROB": round(float(mp), 10)}


def extract_file(
    extractor: Extractor, sonics: SonicsPredictor, path: Path, cache: Path,
    signature: str, deadline: float, parity: bool = False,
) -> RecoveryRecord:
    """Compute one atomic record with exactly one Demucs separation."""
    source_hash = digest(path)
    target = cache / f"{hashlib.sha256((path.stem + source_hash + signature).encode()).hexdigest()}.json"
    if target.is_file():
        record = json.loads(target.read_text(encoding="utf-8"))
        required = {"signature", "source_sha256", "feature", "candidates", "timings"}
        if not required.issubset(record) or record["signature"] != signature or record["source_sha256"] != source_hash:
            raise RecoveryError(f"incomplete or stale recovery cache: {target}")
        if parity and "parity" not in record:
            raise RecoveryError(f"baseline parity missing from cached record: {target}")
        return record
    if time.time() >= deadline:
        raise TimeoutError(f"deadline reached before required file: {path.name}")
    base, device = extractor.base, extractor.device
    audio = base.load_audio(path)
    panns_value, panns_seconds = synchronized(device, lambda: extractor.panns.inference(base.make_panns_segments(audio)))
    panns_predictions = panns_value[0]
    presence = np.stack((panns_predictions[:, extractor.vi].max(1), panns_predictions[:, extractor.mi].max(1)), 1)
    stems, separation_seconds = synchronized(device, lambda: base.separate_voice_and_music(path, extractor.demucs, device))
    voice_audio, music_audio = stems
    voice_fake, voice_seconds = synchronized(device, lambda: extractor.fake_windows(voice_audio))
    music_fake, music_seconds = synchronized(device, lambda: extractor.fake_windows(music_audio))
    raw_result, raw_seconds = synchronized(device, lambda: sonics.predict(audio))
    stem_result, stem_seconds = synchronized(device, lambda: sonics.predict(music_audio))
    g0 = _g0(path.stem, presence, voice_fake, music_fake)
    candidates = candidate_predictions(g0, raw_result["mean_probability"], stem_result["mean_probability"])
    feature = {"ID": path.stem, "duration_seconds": audio.size / 16000, "presence": presence.tolist(),
               "voice_fake_windows": voice_fake, "music_fake_windows": music_fake,
               "sonics_raw": raw_result, "sonics_stem": stem_result}
    record = {"signature": signature, "source_sha256": source_hash, "feature": feature,
              "candidates": candidates, "timings": {"panns": panns_seconds, "separation": separation_seconds,
              "df_voice": voice_seconds, "df_music": music_seconds, "sonics_raw": raw_seconds,
              "sonics_stem": stem_seconds}}
    if parity:
        vp, mp = base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
        vf = base.predict_fake(extractor.df, extractor.fake_index, voice_audio, device)
        mf = base.predict_fake(extractor.df, extractor.fake_index, music_audio, device)
        expected = [base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp]
        actual = [float(g0[key]) for key in ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
                                             "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB")]
        delta = max(abs(left - right) for left, right in zip(expected, actual, strict=True))
        if delta > 1e-5:
            raise AssertionError(f"G0 baseline parity failed for {path.name}: {delta}")
        record["parity"] = {"max_probability_delta": delta, "expected": expected, "actual": actual}
    atomic_json(target, record)
    return record


def manifest_paths(root: Path, rows: Sequence[Mapping[str, str]], expected: int | None = None) -> list[Path]:
    if expected is not None and len(rows) != expected:
        raise RecoveryError(f"required manifest coverage is {expected}, got {len(rows)}")
    if len({row["ID"] for row in rows}) != len(rows):
        raise RecoveryError("manifest IDs must be unique")
    paths = [root / row["path"] for row in rows]
    if any(path.stem != row["ID"] or not path.is_file() for path, row in zip(paths, rows, strict=True)):
        raise RecoveryError("manifest ID/path mismatch or missing audio")
    return paths


def predict_all(extractor: Extractor, sonics: SonicsPredictor, paths: Sequence[Path], cache: Path,
                signature: str, deadline: float, parity_ids: set[str] | None = None) -> tuple[list[dict], dict[str, list[dict]]]:
    records = []
    for index, path in enumerate(paths, 1):
        started = time.perf_counter()
        records.append(extract_file(extractor, sonics, path, cache, signature, deadline,
                                    parity=parity_ids is not None and path.stem in parity_ids))
        print(f"RECOVERY_FEATURE {index}/{len(paths)} {path.stem} {time.perf_counter() - started:.3f}s", flush=True)
    predictions = {name: [record["candidates"][name] for record in records] for name in CANDIDATES}
    return records, predictions


def main() -> None:
    parser = argparse.ArgumentParser()
    for name in ("data", "fresh", "baseline", "assets", "output"):
        parser.add_argument(f"--{name}", required=True, type=Path)
    parser.add_argument("--deadline", required=True, type=float)
    args = parser.parse_args()
    args.output.mkdir(parents=True, exist_ok=True)
    started = time.time()
    development_path = args.data / "development/manifest.csv"
    old_path = args.data / "locked_test/manifest.csv"
    dev = [row for row in read_csv(development_path) if row["split"] == "dev_select"]
    old = read_csv(old_path)
    if len(dev) != 122 or sum(row["split"] == "test_generator" for row in old) != 176 \
            or sum(row["split"] == "test_source" for row in old) != 98:
        raise RecoveryError("old evaluation requires dev_select=122, test_generator=176, test_source=98")
    old_paths = manifest_paths(args.data, [*dev, *old], 396)
    extractor = Extractor(args.baseline / "script.py", include_raw=False)
    try:
        sonics = SonicsPredictor(args.assets, str(extractor.device))
        signature, source_hashes = runtime_signature(extractor, args.baseline, args.assets)
        parity_ids = {path.stem for path in old_paths[:2]}
        old_records, old_predictions = predict_all(extractor, sonics, old_paths, args.output / "cache",
                                                    signature, args.deadline, parity_ids)
        atomic_json(args.output / "raw_features.json", [record["feature"] for record in old_records])
        for name in CANDIDATES:
            write_csv(args.output / f"{name}_predictions.csv", old_predictions[name])
        selection = evaluate_old(dev, old, old_predictions)
        atomic_json(args.output / "selection.json", selection)
        fresh_manifest = args.fresh / "manifest.csv"
        fresh_hash = digest(fresh_manifest)
        lock = {"selected": selection["selected"], "candidate_order": CANDIDATES,
                "formulas": {"music": "G0|max SONICS mean|0.5 blend", "file": "max(voice_present*voice_fake,music_present*music_fake)"},
                "thresholds": {"diagnostic_gain": .005, "eer_regression": .02, "auc_regression": .01,
                               "bootstrap_win_rate": .70, "bootstrap_p10": -.005, "bootstrap_resamples": 2000},
                "source_hashes": source_hashes, "development_manifest_sha256": digest(development_path),
                "old_manifest_sha256": digest(old_path), "fresh_manifest_sha256": fresh_hash,
                "selection_sha256": digest(args.output / "selection.json")}
        lock_hash = lock_candidate(args.output / "candidate_lock.json", lock)
        atomic_json(args.output / "candidate_lock_written.json", {"unix": time.time(),
                    "candidate_lock_sha256": lock_hash})
        if digest(fresh_manifest) != fresh_hash:
            raise RecoveryError("fresh manifest changed after candidate lock")
        atomic_json(args.output / "fresh_opened.json", {"unix": time.time(),
                    "candidate_lock_sha256": lock_hash, "fresh_manifest_sha256": fresh_hash})
        fresh_rows = read_csv(fresh_manifest)
        if any(row["split"] != "fresh_challenge" for row in fresh_rows):
            raise RecoveryError("fresh manifest contains a non-quality row")
        fresh_paths = manifest_paths(args.fresh, fresh_rows)
        durations = [sf.info(path).frames / sf.info(path).samplerate for path in fresh_paths]
        if any(seconds < 4 or seconds > 60.01 for seconds in durations):
            raise RecoveryError("fresh audio duration must be within 4-60 seconds")
        fresh_records, fresh_all = predict_all(extractor, sonics, fresh_paths, args.output / "fresh_cache",
                                                signature, args.deadline)
        atomic_json(args.output / "fresh_raw_features.json", [record["feature"] for record in fresh_records])
        selected = str(selection["selected"])
        fresh_predictions = {name: fresh_all[name] for name in (("G0",) if selected == "G0" else ("G0", selected))}
        for name, values in fresh_predictions.items():
            write_csv(args.output / f"{name}_fresh_predictions.csv", values)
        comparison = decide_fresh(args.output / "candidate_lock.json", fresh_manifest, fresh_rows, fresh_predictions)
        atomic_json(args.output / "comparison.json", comparison)
        stress_audio = extractor.base.load_audio(old_paths[0])
        stress_audio = np.tile(stress_audio, 960000 // stress_audio.size + 1)[:960000]
        stress_path = args.output / "stress_60s.wav"
        sf.write(stress_path, stress_audio, 16000)
        stress = extract_file(extractor, sonics, stress_path, args.output / "stress_cache", signature, args.deadline)
        first_audio, second_audio = extractor.base.load_audio(old_paths[0]), extractor.base.load_audio(old_paths[1])
        before = sonics.predict(first_audio)["mean_probability"]
        sonics.predict(second_audio)
        after = sonics.predict(first_audio)["mean_probability"]
        stress_trace = stress["feature"]["sonics_raw"]
        starts = stress_trace["starts"]
        continuous = starts[0] == 0 and all(right <= left + 80000 for left, right in pairwise(starts))
        if stress_trace["covered_samples"] != 960000 or starts[-1] + 80000 < 960000 or not continuous:
            raise AssertionError("60-second SONICS coverage is incomplete")
        if abs(before - after) > 1e-5:
            raise AssertionError("SONICS prediction changed with file order")
        coverage = {"old_complete": len(old_records) == 396, "fresh_complete": len(fresh_records) == len(fresh_rows),
                    "baseline_parity": [record["parity"] for record in old_records[:2]],
                    "stress_60s": stress_trace, "file_order_max_delta": abs(before - after),
                    "timings": {key: sum(record["timings"][key] for record in [*old_records, *fresh_records])
                                for key in old_records[0]["timings"]}}
        atomic_json(args.output / "coverage_timing.json", coverage)
        atomic_json(args.output / "finished.json", {"status": "finished", "elapsed_seconds": time.time() - started,
                    "candidate_lock_sha256": lock_hash, "development_selected": selection["development_selected"],
                    "old_veto_selected": selection["selected"], "final": comparison["final"],
                    "submission_status": "not_submitted"})
    finally:
        extractor.close()


if __name__ == "__main__":
    main()


In [ ]:
INPUT_ARCHIVES = [{'name': 'v2_validation.zip', 'sha256': 'f08eef8f309fab3a176a89fe34fffeaf687fc99ffe4f16c60157addaf9796b9e', 'bytes': 217713960, 'parts': [{'name': 'v2_validation.zip.part00', 'sha256': '71ad85610171ddbab1f37bd6a20f13c6adfa5886b44d408a8983f645520876b0', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part00'}, {'name': 'v2_validation.zip.part01', 'sha256': 'a357bb091f4e5e4302e1c5b4ce33492850d25714d2c6d66593b80a5ca1e97a04', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part01'}, {'name': 'v2_validation.zip.part02', 'sha256': '899c0f4a3352da01d642417711ca8575c7328e1b6ec7072737edc0415a942444', 'bytes': 49941800, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part02'}]}, {'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l', 'parts': [{'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l'}]}, {'name': 'recovery_validation.zip', 'bytes': 95530127, 'sha256': '0d3e7d2d5bf1509d941b4651a258938d9eab2bedb4ce2b7a64c182c35523dae0', 'drive_id': '1b67mzwMIdH6XP1xHNMvwhqidZnQf_3h5', 'parts': [{'name': 'recovery_validation.zip', 'bytes': 95530127, 'sha256': '0d3e7d2d5bf1509d941b4651a258938d9eab2bedb4ce2b7a64c182c35523dae0', 'drive_id': '1b67mzwMIdH6XP1xHNMvwhqidZnQf_3h5'}]}]

In [ ]:
try:
    (RUN/'code').mkdir()
    for source in WORK.glob('*.py'):
        shutil.copy2(source,RUN/'code'/source.name)
    def sha(path):
        with path.open('rb') as stream:
            return hashlib.file_digest(stream,'sha256').hexdigest()
    (RUN/'source_hashes.json').write_text(json.dumps({p.name:sha(p) for p in (RUN/'code').glob('*.py')},indent=2))
    (RUN/'input_archives.json').write_text(json.dumps(INPUT_ARCHIVES,indent=2))
    packages = ['demucs==4.0.1','panns-inference==0.1.1','transformers==4.57.6',
        'librosa==0.10.2.post1','diffq==0.2.4','einops==0.8.1','timm==1.0.15',
        'scikit-learn==1.8.0','soundfile==0.12.1','safetensors','fvcore']
    with (RUN/'install.log').open('w') as log:
        subprocess.run([sys.executable,'-m','pip','install','-q',*packages],
            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=600)
    import torch, importlib.metadata
    if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name():
        raise RuntimeError('This bounded comparison requires the requested A100 runtime')
    (RUN/'environment.json').write_text(json.dumps({'python':sys.version,'gpu':torch.cuda.get_device_name(),
        'packages':{k:importlib.metadata.version(k) for k in ['torch','torchaudio','torchvision','transformers','timm','numpy','scipy','librosa','scikit-learn']}}))
    for archive in INPUT_ARCHIVES:
        destination = WORK/archive['name']
        with destination.open('wb') as output:
            for part in archive['parts']:
                source = DRIVE/part['name']
                if sha(source) != part['sha256']:
                    raise ValueError(f'Drive part checksum mismatch: {source.name}')
                with source.open('rb') as incoming:
                    shutil.copyfileobj(incoming,output,8*1024**2)
        if sha(destination) != archive['sha256']:
            raise ValueError(f'Archive checksum mismatch: {destination.name}')
        with zipfile.ZipFile(destination) as packed:
            if any(Path(m.filename).is_absolute() or '..' in Path(m.filename).parts for m in packed.infolist()):
                raise ValueError('Unsafe archive path')
            packed.extractall(WORK)
        print('VERIFIED_INPUT',destination.name,flush=True)
    shutil.copy2(DRIVE/'open.zip',WORK/'open.zip')
    with zipfile.ZipFile(WORK/'open.zip') as packed:
        packed.extractall(WORK/'official')
    original = next((WORK/'official').rglob('baseline_submit.zip'))
    if sha(original) != 'cc86f143915c95fd958a0066cbf14b0db9f4c41fd213310c3650b9fe870d0c70':
        raise ValueError('Original baseline archive changed')
    BASELINE = WORK/'baseline'
    with zipfile.ZipFile(original) as packed:
        packed.extractall(BASELINE)
    public = next((WORK/'official').rglob('sample_submission.csv')).parent
    shutil.copytree(public,BASELINE/'data',dirs_exist_ok=True)
    os.chdir(WORK)
    env = dict(os.environ, PYTHONPATH=str(WORK/'sonics_vendor'), HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    command = [sys.executable,'-u','run_recovery.py','--data',str(WORK/'v2_validation'),
        '--fresh',str(WORK/'recovery_validation'),'--baseline',str(BASELINE),
        '--assets',str(WORK/'sonics_assets'),'--output',str(RUN),'--deadline',str(DEADLINE)]
    with (RUN/'experiment.log').open('w',buffering=1) as log:
        process = subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,env=env)
        while process.poll() is None:
            if time.time() >= DEADLINE-60:
                process.terminate()
                process.wait(timeout=30)
                raise TimeoutError('Recovery experiment reached allocation limit')
            time.sleep(15)
            lines = (RUN/'experiment.log').read_text().splitlines()
            print(lines[-1][:800] if lines else 'Loading frozen models...',flush=True)
        if process.returncode:
            raise RuntimeError((RUN/'experiment.log').read_text()[-12000:])
    print('RESULT',json.loads((RUN/'finished.json').read_text()))
    print('DRIVE_RESULTS',RUN)
except Exception:
    (RUN/'failure.txt').write_text(traceback.format_exc())
    raise
finally:
    (RUN/'release_requested.json').write_text(json.dumps({'unix':time.time(),'elapsed_seconds':time.time()-SESSION_START}))
    watchdog.cancel()
    runtime.unassign()